# 09 — Gold Facts

**Purpose:** Build analytical fact tables in the CommerceAI Gold layer using embedded Spark SQL.

**Inputs:**
- `commerceai.silver.sl_orders`
- `commerceai.silver.sl_order_items`
- `commerceai.silver.sl_payments`

**Outputs:**
- `commerceai.gold.fct_orders`

**Main Responsibilities:**
- Preserve one row per order.
- Aggregate order items and payments independently.
- Preserve financial differences and missing source relationships.
- Create historical delivery performance metrics.
- Use Delta tables for analytical consumption.

**Execution / Validation Notes:**
- Run after the Silver validation gate and Gold dimensions.
- Execute transformations using `spark.sql()`.
- Do not join item-level and payment-level records directly.
- Keep monetary values as exact decimals.
- Apply date-based delivery comparisons without assuming source timezone.
- Run `10_gold_validation` separately after building the facts.
- The current implementation uses full-load replacement.

In [0]:
# Keep date interpretation consistent with the existing Gold dimensions.

session_timezone = spark.conf.get("spark.sql.session.timeZone")

print(f"Spark session time zone: {session_timezone}")

if session_timezone not in {"UTC", "Etc/UTC"}:
    raise ValueError(
        "Review the Spark session time zone before building "
        "date-based Gold metrics."
    )

In [0]:

# Grain: one row per order_id.
# Item and payment aggregates must remain independent.

spark.sql("""
CREATE OR REPLACE TABLE commerceai.gold.fct_orders
USING DELTA
COMMENT 'Order-level analytical fact with independent financial aggregates and historical delivery metrics.'
AS

WITH item_aggregates AS (

    SELECT
        order_id,
        COUNT(*) AS item_count,
        COUNT(DISTINCT product_id) AS distinct_product_count,
        COUNT(DISTINCT seller_id) AS distinct_seller_count,

        SUM(price) AS item_value_brl,
        SUM(freight_value) AS freight_value_brl,
        SUM(price + freight_value) AS ordered_total_value_brl

    FROM commerceai.silver.sl_order_items
    GROUP BY order_id

),

payment_aggregates AS (

    SELECT
        order_id,
        COUNT(*) AS payment_count,
        SUM(payment_value) AS paid_value_brl

    FROM commerceai.silver.sl_payments
    GROUP BY order_id

),

order_enriched AS (

    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,

        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        CAST(o.order_purchase_timestamp AS DATE)
            AS order_purchase_date,

        CAST(o.order_estimated_delivery_date AS DATE)
            AS estimated_delivery_date,

        CAST(o.order_delivered_customer_date AS DATE)
            AS actual_delivery_date,

        CAST(
            DATE_FORMAT(o.order_purchase_timestamp, 'yyyyMMdd')
            AS INT
        ) AS order_purchase_date_key,

        CAST(
            DATE_FORMAT(o.order_estimated_delivery_date, 'yyyyMMdd')
            AS INT
        ) AS estimated_delivery_date_key,

        CAST(
            DATE_FORMAT(o.order_delivered_customer_date, 'yyyyMMdd')
            AS INT
        ) AS actual_delivery_date_key,

        (i.order_id IS NOT NULL) AS has_order_items,
        (p.order_id IS NOT NULL) AS has_payments,

        COALESCE(i.item_count, 0) AS item_count,
        COALESCE(i.distinct_product_count, 0)
            AS distinct_product_count,
        COALESCE(i.distinct_seller_count, 0)
            AS distinct_seller_count,
        COALESCE(p.payment_count, 0) AS payment_count,

        i.item_value_brl,
        i.freight_value_brl,
        i.ordered_total_value_brl,
        p.paid_value_brl,

        CASE
            WHEN i.order_id IS NOT NULL
             AND p.order_id IS NOT NULL
            THEN p.paid_value_brl - i.ordered_total_value_brl
        END AS payment_difference_brl,

        CASE
            WHEN i.order_id IS NOT NULL
             AND p.order_id IS NOT NULL
            THEN ABS(
                p.paid_value_brl - i.ordered_total_value_brl
            ) > CAST(0.01 AS DECIMAL(18, 2))
        END AS payment_item_mismatch_flag,

        (
            o.order_status = 'delivered'
            AND o.order_delivered_customer_date IS NOT NULL
            AND o.order_estimated_delivery_date IS NOT NULL
        ) AS is_delivery_eligible,

        (
            o.order_status = 'delivered'
            AND o.order_delivered_customer_date IS NULL
        ) AS delivered_without_delivery_date,

        CASE
            WHEN o.order_delivered_carrier_date IS NOT NULL
             AND o.order_delivered_customer_date IS NOT NULL
            THEN o.order_delivered_carrier_date >
                 o.order_delivered_customer_date
        END AS carrier_after_customer_delivery_flag

    FROM commerceai.silver.sl_orders AS o

    LEFT JOIN item_aggregates AS i
        ON o.order_id = i.order_id

    LEFT JOIN payment_aggregates AS p
        ON o.order_id = p.order_id
)

SELECT
    order_id,
    customer_id,
    order_status,

    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    order_purchase_date,
    estimated_delivery_date,
    actual_delivery_date,

    order_purchase_date_key,
    estimated_delivery_date_key,
    actual_delivery_date_key,

    has_order_items,
    has_payments,

    item_count,
    distinct_product_count,
    distinct_seller_count,
    payment_count,

    item_value_brl,
    freight_value_brl,
    ordered_total_value_brl,
    paid_value_brl,

    payment_difference_brl,
    payment_item_mismatch_flag,

    is_delivery_eligible,

    CASE
        WHEN is_delivery_eligible
        THEN actual_delivery_date > estimated_delivery_date
    END AS is_late_delivery,

    CASE
        WHEN is_delivery_eligible
        THEN GREATEST(
            DATEDIFF(actual_delivery_date, estimated_delivery_date),
            0
        )
    END AS days_late,

    CASE
        WHEN order_status = 'delivered'
         AND actual_delivery_date IS NOT NULL
         AND order_purchase_date IS NOT NULL
        THEN DATEDIFF(actual_delivery_date, order_purchase_date)
    END AS delivery_duration_days,

    CASE
        WHEN estimated_delivery_date IS NOT NULL
         AND order_purchase_date IS NOT NULL
        THEN DATEDIFF(
            estimated_delivery_date,
            order_purchase_date
        )
    END AS promised_delivery_days,

    delivered_without_delivery_date,
    carrier_after_customer_delivery_flag

FROM order_enriched
""")

print("Gold order fact built: commerceai.gold.fct_orders")


In [0]:
display(spark.sql("""
SELECT
    order_id,
    customer_id,
    order_status,
    item_count,
    distinct_seller_count,
    payment_count,
    ordered_total_value_brl,
    paid_value_brl,
    payment_difference_brl,
    is_delivery_eligible,
    is_late_delivery,
    days_late
FROM commerceai.gold.fct_orders
ORDER BY order_purchase_timestamp, order_id
LIMIT 10
"""))